# 03 Character-level CNN (Model 2)

The CNN of Gregório et al. (2024), Figure 1: a character embedding, six convolutions with max-pooling, and a dense classifier. The architecture is in `src/models/charcnn.py`, and the model is scored with the same metrics as the baselines (`src/models/metrics.py`).

It is run in two setups, like Model 1: **binary** (target DGA/legitimate) and **family-first** (target `legit` or a DGA family, collapsed with P(DGA) = 1 - P(legit)). The input is the full domain name as it is, TLD included (nothing is split off), and both setups use the sample weights. Metrics are computed on the DGA/legitimate decision, on the validation set; the test set is kept for the final results.

After loading and encoding the data, the notebook follows four steps:

1. **Define a predictor**: the CNN, which gives each domain a probability for every class.
2. **Define a loss**: weighted cross-entropy plus L1 and L2 penalties on the weights.
3. **Optimize the loss**: Adam on mini-batches, keeping the best epoch on validation.
4. **Test the model**: metrics on the DGA/legitimate decision.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

## Settings

Training follows the paper (Table 2): Adam, learning rate 1e-4, batch size 250, L1 1e-5 and L2 1e-4, 20 epochs, keeping the best epoch on validation (here by macro-F1). On a CPU one epoch over the full training set takes more than an hour, so use a GPU (e.g. Colab) for `SAMPLE = None`.

In [ ]:
SAMPLE = 100_000   # None = full training set
EPOCHS = 20
LR = 1e-4          # Adam learning rate
BATCH_SIZE = 250
L1, L2 = 1e-5, 1e-4   # penalties on the convolution and dense weights
SEED = 0
EVAL_SET = "val"   # "test" only for the final results
WEIGHTED = True    # use the sample weights in the loss
CHECKPOINTS = ROOT / "checkpoints"   # trained models are saved here (not committed)

In [ ]:
import copy
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import BatchSampler, DataLoader, RandomSampler, TensorDataset

from src.data.dataset import load_splits, sample_weights
from src.models.charcnn import CharCNN
from src.models.encoding import class_codes, encode
from src.models.metrics import binary_metrics, recall_per_family
from src.models.train import p_dga, penalty, pick_device, predict_proba, set_seed

device = pick_device()
print("device:", device, torch.cuda.get_device_name() if device.type == "cuda" else "")

train_full, val, test = load_splits()
train = train_full.sample(n=SAMPLE, random_state=SEED) if SAMPLE else train_full
evaluate = test if EVAL_SET == "test" else val
y_val = (val["label"] == "dga").astype(int).to_numpy()
y_eval = (evaluate["label"] == "dga").astype(int).to_numpy()
print(f"train: {len(train):,} rows (of {len(train_full):,}), "
      f"{EVAL_SET}: {len(evaluate):,} rows")

## Input encoding

The full domain name, TLD included, is the input. Each character becomes its ASCII code, padded with 0 at the end to 93 characters, the longest domain in the dataset, so no domain is cut (the paper used 70, the longest in its data). Classes are numbered from the full training set, so every family has an output even if a sample misses it.

In [ ]:
x_train, x_val, x_eval = (encode(part["domain"]) for part in (train, val, evaluate))
classes = {"label": sorted(train_full["label"].unique()),
           "family": sorted(train_full["family"].unique())}
print(f"input: {x_train.shape}, {len(classes['family'])} family classes")
print(train["domain"].iloc[0], "->", x_train[0][:20], "...")

## 1. Define a predictor

The predictor is the CNN of Figure 1. It maps an encoded domain to one score (logit) per class, and a softmax turns the scores into probabilities. The classes are `dga` and `legit` in the binary setup, and `legit` plus the DGA families in the family-first setup. In both, the DGA probability is P(DGA) = 1 - P(legit).

With a 93-character input the flattened size is 93 x 30 = 2790 (70 x 30 = 2100 in the paper).

In [ ]:
SETUPS = {"binary": "label", "family": "family"}   # setup -> target column


def make_predictor(task):
    """A new, untrained CNN with one output per class of `task`."""
    return CharCNN({task: len(classes[task])})


def predict_p_dga(model, x, task):
    """P(DGA) = 1 - P(legit), from the softmax over the classes of `task`."""
    return p_dga(predict_proba(model, x), task, classes)


model = make_predictor("family")
print(model)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

Before training, the weights are random: the predictor gives almost the same P(DGA) to every domain, so it cannot yet tell DGA from legitimate.

In [ ]:
examples = val.groupby("label").head(3)
model = make_predictor("label")
pd.DataFrame({"domain": examples["domain"].to_numpy(),
              "label": examples["label"].to_numpy(),
              "P(DGA), untrained": predict_p_dga(model, encode(examples["domain"]), "label").round(3)})

## 2. Define a loss

The loss of a batch of $B$ domains is the paper's loss (Table 2) with the sample weights added:

$$L = \frac{1}{B}\sum_{i=1}^{B} w_i \,\big(-\log P(y_i \mid x_i)\big) \;+\; \lambda_1 \sum |W| \;+\; \lambda_2 \sum W^2$$

- The first term is the cross-entropy: minus the log of the probability the predictor gives to the true class $y_i$ of domain $x_i$, weighted by its sample weight $w_i$. With `WEIGHTED = False`, every $w_i$ is 1.
- The other two terms are the L1 ($\lambda_1$ = 1e-5) and L2 ($\lambda_2$ = 1e-4) penalties on the convolution and dense weights $W$ (not the embedding, not the biases), added as Keras does (`penalty` in `src/models/train.py`).

In [ ]:
def loss_fn(model, logits, y, w, task):
    """Loss of one batch: weighted cross-entropy + L1/L2 penalty on the weights."""
    cross_entropy = F.cross_entropy(logits[task], y, reduction="none")   # -log P(y_i | x_i)
    return (cross_entropy * w).mean() + penalty(model, L1, L2)

The loss of the untrained binary predictor on the first batch of the training set:

In [ ]:
xb = torch.as_tensor(x_train[:BATCH_SIZE])
yb = torch.as_tensor(class_codes(train["label"].iloc[:BATCH_SIZE], classes["label"]))
wb = torch.as_tensor(np.array(train["weight"].iloc[:BATCH_SIZE]) if WEIGHTED
                     else np.ones(BATCH_SIZE), dtype=torch.float32)
with torch.no_grad():
    loss = loss_fn(model, model(xb), yb, wb, "label")
    reg = penalty(model, L1, L2)
print(f"loss {loss:.4f} = weighted cross-entropy {loss - reg:.4f} + penalty {reg:.4f}")

## 3. Optimize the loss

Adam (learning rate `LR`) minimizes the loss on shuffled mini-batches of `BATCH_SIZE` domains, for `EPOCHS` epochs. After every epoch, the predictor is scored on the validation set, and the weights of the epoch with the highest macro-F1 are kept (the paper keeps the best accuracy).

Two losses are recorded per epoch, both with `loss_fn`:

- $J_{train}$: the mean loss over the epoch's training batches (as Keras reports it), so it is measured while the weights are still changing.
- $J_{val}$: the loss on the whole validation set at the end of the epoch. The validation sample weights are computed in the same way as the training ones (`sample_weights`, from the validation family sizes), so the two losses are on the same scale.

In [ ]:
w_val = torch.as_tensor(sample_weights(val) if WEIGHTED else np.ones(len(val)),
                        dtype=torch.float32)


@torch.no_grad()
def validate(model, task, batch_size=4096):
    """J_val (loss_fn on the whole validation set) and the validation P(DGA)."""
    model.eval()
    y = torch.as_tensor(class_codes(val[task], classes[task]))
    total, proba = 0.0, []
    for start in range(0, len(x_val), batch_size):
        end = start + batch_size
        logits = model(torch.as_tensor(x_val[start:end]).to(device))
        loss = loss_fn(model, logits, y[start:end].to(device), w_val[start:end].to(device), task)
        total += loss.item() * len(logits[task])
        proba.append(logits[task].softmax(1).cpu().numpy())
    return total / len(x_val), p_dga({task: np.concatenate(proba)}, task, classes)


def optimize(model, task):
    """Minimize loss_fn with Adam and keep the weights of the best validation epoch.

    Returns one row per epoch (J_train, J_val and validation metrics) and the
    validation P(DGA) of the kept epoch.
    """
    model.to(device)
    w = np.array(train["weight"]) if WEIGHTED else np.ones(len(train))
    data = TensorDataset(torch.as_tensor(x_train), torch.as_tensor(w, dtype=torch.float32),
                         torch.as_tensor(class_codes(train[task], classes[task])))
    # Shuffled mini-batches, each taken with one indexing step instead of one domain at a
    # time: the same batches as DataLoader(shuffle=True), about 10x less loading time.
    shuffle = torch.Generator().manual_seed(SEED)
    batches = BatchSampler(RandomSampler(data, generator=shuffle), BATCH_SIZE, drop_last=False)
    loader = DataLoader(data, sampler=batches, batch_size=None, generator=shuffle)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history, best_f1, best_state, best_p = [], -1.0, None, None
    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        model.train()
        total = torch.zeros((), device=device)
        for xb, wb, yb in loader:
            xb, wb, yb = xb.to(device), wb.to(device), yb.to(device)
            loss = loss_fn(model, model(xb), yb, wb, task)
            optimizer.zero_grad()
            loss.backward()    # gradient of the loss
            optimizer.step()   # one Adam step
            total += loss.detach() * len(xb)

        val_loss, p = validate(model, task)   # validation, to pick the best epoch
        row = {"epoch": epoch, "train_loss": total.item() / len(x_train), "val_loss": val_loss,
               **binary_metrics(y_val, p), "seconds": round(time.time() - start)}
        history.append(row)
        if row["macro_f1"] > best_f1:
            best_f1, best_state, best_p = row["macro_f1"], copy.deepcopy(model.state_dict()), p
        print(f"epoch {epoch:2d}: J_train {row['train_loss']:.4f}, J_val {row['val_loss']:.4f}, "
              f"val macro-F1 {row['macro_f1']:.4f}, AUC {row['roc_auc']:.4f} "
              f"({row['seconds']} s)")

    model.load_state_dict(best_state)
    return pd.DataFrame(history).set_index("epoch"), best_p

One model per setup, each from the same seed. The model is kept at its best validation epoch and saved to `checkpoints/`.

In [ ]:
CHECKPOINTS.mkdir(exist_ok=True)

models, histories, p_val = {}, {}, {}
for setup, task in SETUPS.items():
    print(f"--- {setup} ({len(classes[task])} classes)")
    set_seed(SEED)
    models[setup] = make_predictor(task)
    histories[setup], p_val[setup] = optimize(models[setup], task)
    torch.save(models[setup].state_dict(), CHECKPOINTS / f"charcnn_{setup}_seed{SEED}.pt")

In [ ]:
# One loss panel per setup (the binary and family losses are on different scales), then
# validation macro-F1. The dotted line marks the kept epoch (best validation macro-F1).
fig, axes = plt.subplots(1, len(histories) + 1, figsize=(15, 3.6))
for ax, (setup, history) in zip(axes, histories.items()):
    ax.plot(history.index, history["train_loss"], marker="o", label="$J_{train}$")
    ax.plot(history.index, history["val_loss"], marker="o", label="$J_{val}$")
    ax.axvline(history["macro_f1"].idxmax(), color="grey", linestyle=":", label="kept epoch")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title(f"Loss, {setup}")
    ax.legend()

ax = axes[-1]
for setup, history in histories.items():
    ax.plot(history.index, history["macro_f1"], marker="o", label=setup)
ax.set_xlabel("epoch")
ax.set_ylabel("macro-F1")
ax.set_title("Validation macro-F1")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Test the model

Each trained predictor gives P(DGA) for every domain of the evaluation set (`EVAL_SET`: the validation set for now, the test set for the final results). A domain is called DGA when P(DGA) >= 0.5. The metrics are those of the baselines, on the DGA/legitimate decision.

In [ ]:
# On the validation set, P(DGA) of the kept epoch was already computed in step 3 (same
# weights, same numbers), so the predictor only needs to run again on the test set.
p_dga_eval = {setup: p_val[setup] if EVAL_SET == "val" else predict_p_dga(models[setup], x_eval, task)
              for setup, task in SETUPS.items()}
results = {("Model 2", setup): binary_metrics(y_eval, p) for setup, p in p_dga_eval.items()}

In [ ]:
table = pd.DataFrame(results).T.round(4)
table.index.names = ["model", "setup"]
table

### Recall per DGA family

Recall of each DGA family in the two setups, families ordered from largest to smallest.

In [ ]:
family_sizes = train_full.loc[train_full["label"] == "dga", "family"].value_counts()
recall = pd.DataFrame({
    setup: recall_per_family(evaluate["family"], y_eval, (p >= 0.5).astype(int))
    for setup, p in p_dga_eval.items()}).reindex(family_sizes.index)
recall.columns = ["binary", "family-first"]

fig, ax = plt.subplots(figsize=(12, 3.6))
x = np.arange(len(recall))
for setup, marker in [("binary", "o"), ("family-first", "D")]:
    ax.scatter(x, recall[setup], s=24, marker=marker, label=setup, zorder=3)
ax.set_xlim(-1, len(recall))
ax.set_ylim(-0.03, 1.05)
ax.set_xlabel(f"DGA family, largest to smallest ({len(recall)} families)")
ax.set_ylabel("recall")
ax.set_title("Recall per DGA family, binary vs. family-first")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

### Families the CNN misses most

The 10 DGA families with the lowest recall in the family-first setup.

In [ ]:
recall.sort_values("family-first").head(10).round(3)